In [2]:
from pydantic import BaseModel
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain.agents import create_tool_calling_agent, AgentExecutor

import os

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
class DefaultResponse(BaseModel):
    response: str
    tool_used: list[str]

model_name = "llama-3.3-70b-versatile"
llm = ChatGroq(
          api_key=os.getenv("GROQ_API_KEY"),
          model_name=model_name,
      )

parser = PydanticOutputParser(pydantic_object=DefaultResponse)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are a research assistant that will help generate a research paper.
            Answer the user query and use neccessary tools.
            Wrap the output in this format and provide no other text\n{format_instructions}
            """,
        ),
        ("placeholder", "{chat_history}"),
        ("human", "{query}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
).partial(format_instructions=parser.get_format_instructions())

tools = []

In [5]:
agent = create_tool_calling_agent(
    llm=llm,
    prompt=prompt,
    tools=tools
)

In [6]:
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
query = "Apa yang bisa kamu lakukan?"
raw_response = agent_executor.invoke({"query": query})
raw_response



> Entering new AgentExecutor chain...
{"response": "Saya dapat membantu Anda dengan tugas-tugas seperti menjawab pertanyaan, memberikan informasi, menerjemahkan teks, dan membantu dalam penelitian.", "tool_used": ["Bahasa Pemrograman", "Kamus", "Mesin Pencari", "Basis Data Pengetahuan"]}

> Finished chain.


{'query': 'Apa yang bisa kamu lakukan?',
 'output': '{"response": "Saya dapat membantu Anda dengan tugas-tugas seperti menjawab pertanyaan, memberikan informasi, menerjemahkan teks, dan membantu dalam penelitian.", "tool_used": ["Bahasa Pemrograman", "Kamus", "Mesin Pencari", "Basis Data Pengetahuan"]}'}

In [ ]:
structured_response = parser.parse(raw_response.get("output"))
print(structured_response.response)